# **Modelo baseline — probabilidad de adjudicacion (Capacidad 3)**

Trabaja sobre `secop_ctei_procesos_deflactado.csv`, filtrado al universo
competitivo (unica poblacion donde `adjudicado` tiene senal real).

Progresion de 3 baselines, de mas tonto a menos tonto:
1. **Trivial**: siempre predecir la clase mayoritaria.
2. **Regla simple**: tasa historica de adjudicacion por modalidad (calculada
   SOLO con train, aplicada a test).
3. **Regresion logistica**: con las features clasificadas como validas al
   momento de publicacion (sin fuga).

Split **temporal** (no aleatorio): train = antes del corte, test = despues.
Cualquier modelo "de verdad" que hagan despues de este notebook tiene que
superar al baseline 3 para justificar la complejidad adicional.

In [14]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                              classification_report)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

# 1. Carga y filtrado al universo competitivo

Mismo criterio que en el notebook de correcciones: fuera de estas
modalidades, `adjudicado` es estructuralmente 0 (no hay senal que
aprender).

In [15]:
MODALIDADES_COMPETITIVAS = [
    "Licitación pública", "Licitación pública Obra Publica",
    "Licitación Pública Acuerdo Marco de Precios",
    "Concurso de méritos abierto", "Concurso de méritos con precalificación",
    "Selección Abreviada de Menor Cuantía",
    "Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes",
    "Selección abreviada subasta inversa", "Mínima cuantía",
    "Contratación Directa (con ofertas)", "Contratación régimen especial (con ofertas)",
]

df = pd.read_csv("secop_ctei_procesos_deflactado.csv", low_memory=False)
df["fecha_de_publicacion_del"] = pd.to_datetime(df["fecha_de_publicacion_del"], errors="coerce")

df = df[df["modalidad_de_contratacion"].isin(MODALIDADES_COMPETITIVAS)].copy()
df = df[df["fecha_de_publicacion_del"].notna()].copy()  # el split temporal exige fecha

print(f"Universo competitivo con fecha valida: {len(df):,} procesos")
print(f"Tasa de adjudicacion: {df['adjudicado_proceso'].mean():.1%}")

# CORRECCION POR CENSURA: un "No" solo es confiable si el proceso
# llego a un estado verdaderamente TERMINAL (Cancelado). Todo lo demas
# que no sea adjudicado=Si ni Cancelado se EXCLUYE (no se re-etiqueda):
# su desenlace real todavia no se conoce (censura por la derecha),
# verificado empiricamente: 85.4% de los "No" en procesos recientes
# seguian en estados abiertos (Evaluacion/Publicado/Abierto/Seleccionado).
ESTADOS_TERMINALES_NEGATIVOS = ["Cancelado"]
antes = len(df)
df = df[
    df["adjudicado_proceso"]
    | df["estado_del_procedimiento"].isin(ESTADOS_TERMINALES_NEGATIVOS)
].copy()
print(f"\nFiltrado por censura: {antes:,} -> {len(df):,} "
      f"({antes - len(df):,} procesos excluidos por seguir abiertos)")
print(f"Tasa de adjudicacion tras filtrar censura: {df['adjudicado_proceso'].mean():.1%}")

Universo competitivo con fecha valida: 69,684 procesos
Tasa de adjudicacion: 55.3%

Filtrado por censura: 69,684 -> 44,512 (25,172 procesos excluidos por seguir abiertos)
Tasa de adjudicacion tras filtrar censura: 86.6%


# 2. Seleccion de features (sin fuga) y target

Solo columnas disponibles **al momento de publicacion**. Quedan fuera
`valor_adjudicado_total(_real)`, `fecha_adjudicacion`,
`n_proveedores_adjudicados`, `respuestas_al_procedimiento` — son el
resultado o se conocen solo al cierre.

`entidad` queda fuera de este baseline por alta cardinalidad (miles de
valores distintos) — es candidata para un modelo mas avanzado con target
encoding calculado solo sobre train; aqui se prioriza simplicidad.

In [16]:
TARGET = "adjudicado_proceso"

FEATURES_NUMERICAS = ["precio_base_real", "duracion", "numero_de_lotes"]
FEATURES_CATEGORICAS = ["modalidad_de_contratacion", "segmento_unspsc"]  # departamento va agrupado (ver mas abajo)

df[TARGET] = df[TARGET].astype(str).isin(["True", "1", "1.0"])

# precio_base_real: filtrar negativos/cero (ya flagueados en el pipeline) y
# aplicar log porque la distribucion de montos es muy sesgada
df = df[df["precio_base_real"].fillna(-1) > 0].copy()
df["log_precio_base_real"] = np.log1p(df["precio_base_real"])
FEATURES_NUMERICAS = ["log_precio_base_real", "duracion", "numero_de_lotes"]

# duracion y numero_de_lotes: imputar con la mediana de TRAIN (se calcula despues del split)
df["mes_publicacion"] = df["fecha_de_publicacion_del"].dt.month
df["anio_publicacion"] = df["fecha_de_publicacion_del"].dt.year

# departamento_entidad: agrupar colas largas para no explotar el one-hot
top_departamentos = df["departamento_entidad"].value_counts().head(15).index
df["departamento_agrupado"] = df["departamento_entidad"].where(
    df["departamento_entidad"].isin(top_departamentos), "Otro"
)

print(f"Filas tras limpiar precio_base_real: {len(df):,}")
print(df[TARGET].value_counts(normalize=True).round(3))

Filas tras limpiar precio_base_real: 44,340
adjudicado_proceso
True     0.866
False    0.134
Name: proportion, dtype: float64


# 3. Split temporal train / test

Mismo corte verificado antes: train/test quedan con tasas de adjudicacion
casi identicas, asi que el corte es representativo.

In [17]:
FECHA_CORTE = "2025-07-01"

train = df[df["fecha_de_publicacion_del"] < FECHA_CORTE].copy()
test = df[df["fecha_de_publicacion_del"] >= FECHA_CORTE].copy()

print(f"Train: {len(train):,} procesos ({train['fecha_de_publicacion_del'].min().date()} a "
      f"{train['fecha_de_publicacion_del'].max().date()}) | tasa adjudicacion: {train[TARGET].mean():.1%}")
print(f"Test:  {len(test):,} procesos ({test['fecha_de_publicacion_del'].min().date()} a "
      f"{test['fecha_de_publicacion_del'].max().date()}) | tasa adjudicacion: {test[TARGET].mean():.1%}")

# imputar duracion/numero_de_lotes con la MEDIANA DE TRAIN (nunca con la de todo el dataset: eso seria fuga)
for col in ["duracion", "numero_de_lotes"]:
    mediana_train = train[col].median()
    train[col] = train[col].fillna(mediana_train)
    test[col] = test[col].fillna(mediana_train)

Train: 33,283 procesos (2022-01-03 a 2025-06-30) | tasa adjudicacion: 86.3%
Test:  11,057 procesos (2025-07-01 a 2026-07-29) | tasa adjudicacion: 87.7%


# 4. Baseline 1 — Trivial (clase mayoritaria)

El piso absoluto: cualquier cosa que hagan despues tiene que superar esto.

In [18]:
clase_mayoritaria = train[TARGET].mode()[0]
pred_trivial = np.full(len(test), clase_mayoritaria)

acc_trivial = accuracy_score(test[TARGET], pred_trivial)
print(f"Baseline 1 (trivial, siempre '{clase_mayoritaria}'): accuracy = {acc_trivial:.1%}")
print("(AUC no aplica: no hay variacion en la prediccion)")

Baseline 1 (trivial, siempre 'True'): accuracy = 87.7%
(AUC no aplica: no hay variacion en la prediccion)


# 5. Baseline 2 — Regla simple (tasa historica por modalidad)

Se calcula la tasa de adjudicacion de CADA modalidad **solo con train**, y
se usa esa tasa (umbral 0.5) para predecir en test. Simula "el sentido
comun de alguien que conoce el dominio", sin ningun modelo estadistico.

In [19]:
tasa_por_modalidad_train = train.groupby("modalidad_de_contratacion")[TARGET].mean()
print("Tasas aprendidas en train:")
print(tasa_por_modalidad_train.round(3))

pred_regla = test["modalidad_de_contratacion"].map(tasa_por_modalidad_train) >= 0.5
pred_regla = pred_regla.fillna(clase_mayoritaria)  # modalidad no vista en train (rara, pero por si acaso)
prob_regla = test["modalidad_de_contratacion"].map(tasa_por_modalidad_train).fillna(train[TARGET].mean())

acc_regla = accuracy_score(test[TARGET], pred_regla)
auc_regla = roc_auc_score(test[TARGET], prob_regla)
print(f"\nBaseline 2 (regla por modalidad): accuracy = {acc_regla:.1%} | AUC = {auc_regla:.3f}")

Tasas aprendidas en train:
modalidad_de_contratacion
Concurso de méritos abierto                                    0.861
Concurso de méritos con precalificación                        0.800
Contratación Directa (con ofertas)                             0.904
Contratación régimen especial (con ofertas)                    0.819
Licitación pública                                             0.855
Licitación pública Obra Publica                                0.860
Mínima cuantía                                                 0.862
Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes    0.839
Selección Abreviada de Menor Cuantía                           0.765
Selección abreviada subasta inversa                            0.902
Name: adjudicado_proceso, dtype: float64

Baseline 2 (regla por modalidad): accuracy = 87.7% | AUC = 0.553


# 6. Baseline 3 — Regresion logistica

Con las features numericas + categoricas definidas en la seccion 2. El
encoder categorico (`get_dummies`) se ajusta sobre train y luego se alinean
las columnas de test (`reindex`) para que una categoria vista solo en test
no rompa el modelo ni se cuele como informacion nueva.

In [20]:
def preparar_X(data, columnas_referencia=None):
    cat = pd.get_dummies(data[FEATURES_CATEGORICAS + ["departamento_agrupado"]],
                          drop_first=True)
    num = data[FEATURES_NUMERICAS + ["mes_publicacion", "anio_publicacion"]].reset_index(drop=True)
    X = pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)
    if columnas_referencia is not None:
        X = X.reindex(columns=columnas_referencia, fill_value=0)
    return X

X_train = preparar_X(train)
X_test = preparar_X(test, columnas_referencia=X_train.columns)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)

modelo = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
modelo.fit(X_train_esc, train[TARGET])

pred_modelo = modelo.predict(X_test_esc)
prob_modelo = modelo.predict_proba(X_test_esc)[:, 1]

acc_modelo = accuracy_score(test[TARGET], pred_modelo)
auc_modelo = roc_auc_score(test[TARGET], prob_modelo)
print(f"Baseline 3 (regresion logistica): accuracy = {acc_modelo:.1%} | AUC = {auc_modelo:.3f}")
print()
print(classification_report(test[TARGET], pred_modelo))

Baseline 3 (regresion logistica): accuracy = 66.1% | AUC = 0.588

              precision    recall  f1-score   support

       False       0.16      0.42      0.23      1364
        True       0.90      0.69      0.78      9693

    accuracy                           0.66     11057
   macro avg       0.53      0.56      0.51     11057
weighted avg       0.80      0.66      0.71     11057



## 6.1 Matriz de confusion y coeficientes (que variables pesan mas)

In [21]:
cm = confusion_matrix(test[TARGET], pred_modelo)
print("Matriz de confusion (filas=real, columnas=prediccion):")
print(pd.DataFrame(cm, index=["Real: No", "Real: Si"], columns=["Pred: No", "Pred: Si"]))

coeficientes = pd.Series(modelo.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
print("\nTop 15 variables con mayor peso (valor absoluto del coeficiente, escala estandarizada):")
print(coeficientes.head(15).round(3))

Matriz de confusion (filas=real, columnas=prediccion):
          Pred: No  Pred: Si
Real: No       575       789
Real: Si      2964      6729

Top 15 variables con mayor peso (valor absoluto del coeficiente, escala estandarizada):
modalidad_de_contratacion_Contratación Directa (con ofertas)             0.140
modalidad_de_contratacion_Selección Abreviada de Menor Cuantía          -0.121
log_precio_base_real                                                     0.116
modalidad_de_contratacion_Contratación régimen especial (con ofertas)   -0.097
modalidad_de_contratacion_Mínima cuantía                                 0.080
modalidad_de_contratacion_Selección abreviada subasta inversa            0.078
departamento_agrupado_Boyacá                                            -0.072
departamento_agrupado_Huila                                             -0.065
numero_de_lotes                                                          0.048
departamento_agrupado_Otro                                

# 7. Comparacion final — la tabla que va al reporte

In [22]:
resumen = pd.DataFrame([
    {"modelo": "1. Trivial (clase mayoritaria)", "accuracy": acc_trivial, "auc": np.nan},
    {"modelo": "2. Regla simple (por modalidad)", "accuracy": acc_regla, "auc": auc_regla},
    {"modelo": "3. Regresion logistica", "accuracy": acc_modelo, "auc": auc_modelo},
])
resumen["accuracy"] = (resumen["accuracy"] * 100).round(1)
resumen["auc"] = resumen["auc"].round(3)
resumen

,modelo,accuracy,auc
0,1. Trivial (clase mayoritaria),87.7,NaN
1,2. Regla simple (por modalidad),87.7,0.553
2,3. Regresion logistica,66.1,0.588


## 7.1 Hallazgos

- **🚩 La correccion por censura cambia el balance de clases una tercera vez, y en
  sentido contrario a lo esperado**: el universo competitivo "crudo" (NB2) tenia 53.8% de
  tasa de adjudicacion; al excluir los procesos que aun siguen abiertos (censura por la
  derecha — 25,172 de 69,684, 36.1% del universo competitivo con fecha), la tasa sube a
  **86.6%**. Es decir, entre los procesos competitivos que **ya llegaron a un desenlace
  observable** (adjudicado o cancelado), la inmensa mayoria termina adjudicada. Esto tiene
  sentido de dominio: una vez se abre una licitacion/concurso competitivo formal, lo
  esperable es que se adjudique; la cancelacion es la excepcion. Recorrido completo del
  desbalance en el proyecto: 7.89% (dataset completo, ver EDA) → 53.8% (universo
  competitivo, ver notebook de correcciones) → 86.6% (universo competitivo + resuelto).
  **Esta ultima cifra es la correcta para entrenar y para reportar** — las dos anteriores
  mezclaban modalidades sin senal o procesos cuyo desenlace todavia no existe.
- **Los baselines 1 y 2 empatan exactamente (87.7% accuracy) porque todas las tasas por
  modalidad estan por encima del umbral 0.5** (rango 0.765-0.904): con ese desbalance, la
  "regla por modalidad" termina prediciendo "Si" para absolutamente todos los casos,
  igual que el baseline trivial. El AUC de la regla (0.553) revela que, aunque la
  *prediccion binaria* colapsa a "siempre Si", el *ranking* por modalidad apenas se
  distingue de aleatorio — la modalidad sola no separa bien quien se cancela.
- **La regresion logistica (baseline 3) baja la accuracy a 66.1% pero es la eleccion
  correcta**, no un retroceso: con `class_weight="balanced"` deja de optimizar accuracy
  bruta (que con este desbalance se gana trivialmente diciendo siempre "Si") y mejora el
  recall real de la clase minoritaria "No" (cancelado) de 0% (baselines 1 y 2, que nunca
  predicen "No") a 42% (575 de 1,364). El costo es precision baja en esa clase (0.16) —
  muchos falsos "No". **Para el reporte: comparar los 3 baselines solo por accuracy es
  enganoso; el AUC (0.588 vs 0.553 vs no aplica) y el recall de la clase minoritaria son
  las metricas que reflejan si el modelo realmente aporta algo.**
- **Poder predictivo global todavia debil**: AUC = 0.588 esta apenas por encima de 0.5
  (aleatorio). Con las features disponibles sin fuga (precio base, duracion, numero de
  lotes, modalidad, segmento, departamento, mes/anio), el modelo distingue algo pero no
  mucho quien termina cancelado dentro del universo competitivo. Es coherente con que las
  variables mas fuertes del EDA (`respuestas_al_procedimiento`) se excluyeron correctamente
  por fuga — el poder predictivo "facil" no esta disponible sin trampas.
- **Variables que mas pesan**: la modalidad domina (Contratacion Directa con ofertas
  suma probabilidad de adjudicacion; Seleccion Abreviada de Menor Cuantia y Contratacion
  regimen especial la restan), `log_precio_base_real` suma (procesos de mayor presupuesto
  tienden a adjudicarse mas), y aparecen **varios departamentos con coeficiente negativo
  notable** (Boyaca, Huila, Risaralda, Norte de Santander, Casanare) — señal nueva, no
  vista en los notebooks anteriores, que apunta a que ciertas regiones tienen tasas de
  cancelacion sistematicamente mas altas dentro del universo competitivo. Vale la pena
  profundizar esto en un notebook de cierre (tabla de tasa de cancelacion por
  departamento) antes de afirmarlo con fuerza en el reporte.
- **Siguiente paso logico, no hecho aqui**: agregar features historicas de entidad/
  proveedor calculadas solo con datos previos a cada proceso (mencionado como pendiente
  en el notebook de correcciones, seccion D.2) — es plausible que suban el AUC bastante
  mas que las features puramente estructurales usadas en este baseline.

## 7.2 Verificacion del gap de deflactacion senalado en el notebook anterior

El notebook de correcciones (`Correcciones_outliers_y_modelado.ipynb`, seccion D.3)
advirtio que `secop_ctei_procesos_deflactado.csv` se genero **sin** aplicar el filtro
`flag_valor_implausible`, por lo que el outlier de EAG podia colarse en las columnas
`_real`. Revisando el uso concreto en este notebook: la unica columna `_real` usada como
feature es `precio_base_real` (via `log_precio_base_real`), **no**
`valor_adjudicado_total_real`. El `precio_base` original de la fila de EAG
(463,454,668 COP) es plausible por si mismo — el outlier estaba solo en
`valor_total_adjudicacion`, no en `precio_base` — asi que este baseline en particular no
se ve afectado por el gap. **Aun asi, el gap sigue sin corregirse en el archivo fuente**:
cualquier notebook futuro que use `valor_adjudicado_total_real` de
`secop_ctei_procesos_deflactado.csv` (por ejemplo para reportar montos totales o series
de valor en el reporte final) debe refiltrar por `flag_valor_implausible` antes de usarla,
o regenerar el CSV aplicando el filtro antes de deflactar.

## 7.3 Limitaciones y proximos pasos

- El filtro de censura (solo `adjudicado=Si` o `estado="Cancelado"`) descarta 36.1% del
  universo competitivo. Es la decision metodologicamente correcta para evitar mal-etiquetar
  "aun no se sabe" como "No", pero implica que el modelo **no predice el desenlace de
  procesos recien publicados que siguen abiertos** (que es, en la practica, el caso de uso
  mas util de un producto real: predecir *mientras* el proceso esta abierto). Documentar
  esta limitacion explicitamente en el reporte: el baseline responde "de los procesos que
  ya se resolvieron, cual fue el patron", no "va a adjudicarse este proceso que acabo de
  publicarse" — para eso ultimo se necesitaria un enfoque de supervivencia/censura
  explicito (ej. modelo de riesgo competitivo cancelado-vs-adjudicado con tiempo hasta el
  evento), no una clasificacion binaria simple.
- `entidad` se dejo fuera del baseline por alta cardinalidad. Dado que el notebook de
  correcciones (D.2) ya identifico esta columna como candidata a un encoding calculado
  solo sobre train, y que en 3.4 del EDA se vio que la relacion proveedor-entidad es muy
  poco diversificada (75% de proveedores le venden a una sola entidad), es probable que
  una feature de "historial pasado de esta entidad/este tipo de proceso" (calculada con
  ventana estrictamente anterior a la fecha del proceso) aporte mas señal que
  precio/duracion/lotes solos.
- No se probo ningun modelo no lineal (arbol/boosting) que pueda capturar interacciones
  entre modalidad, departamento y presupuesto sin necesidad de un one-hot tan disperso —
  candidato natural para el siguiente notebook si el AUC de 0.588 se considera insuficiente
  para el reporte final.